# CodePause Phase 2: Quality & Scale

This notebook implements the Phase 2 workflow focusing on solving the evaluation gap and scaling with QLoRA on Qwen 1.5B.

## 1. Setup Environment

In [ ]:
!pip uninstall -y -q torchao || true
!pip install -q transformers peft bitsandbytes datasets accelerate trl scipy pytest pytest-cov pyyaml

import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git"
BRANCH = "main"
REPO_DIR = "/content/amdh"


def sh(cmd, cwd=None):
    print("$ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)


if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    sh(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
sh(["git", "log", "--oneline", "-1"])


def run_step(name, cmd, cwd=None):
    print(f"\n=== {name} - START ===", flush=True)
    print(" ".join(cmd), flush=True)
    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in proc.stdout:
        output_lines.append(line)
        print(line, end="", flush=True)
        sys.stdout.flush()
    return_code = proc.wait()
    elapsed = time.time() - start
    print(f"\n=== {name} - END ({elapsed:.1f}s, rc={return_code}) ===", flush=True)
    if return_code != 0:
        tail = "".join(output_lines[-40:])
        raise RuntimeError(f"HARD STOP: {name} failed with exit code {return_code}\n{tail}")
    return "".join(output_lines)


In [ ]:
# Cell 1b: Generate metadata provenance
import json
import subprocess
import torch

# Detect hardware
if torch.cuda.is_available():
    hardware = torch.cuda.get_device_name(0)
else:
    hardware = "cpu"

# Get git SHA
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except:
    git_sha = "unknown"

# Build metadata dict
metadata = {
    "phase": "2",
    "model_name": "Qwen/Qwen1.5-1.8B-Chat",
    "hardware": hardware,
    "timestamp": subprocess.check_output(["date", "-Iseconds"], text=True).strip(),
    "git_sha": git_sha,
    "notebook": "codepause_phase_2_quality_scale.ipynb"
}

# Write metadata JSON
os.makedirs("results", exist_ok=True)
with open("results/metadata_run.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata generated: {metadata}")


In [ ]:
# Cell 1c: Mock dry-run - validates CLI wiring before GPU eval
run_step(
    "Mock dry-run validation",
    [
        "python",
        "eval/evaluate_baseline.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--problems_path",
        "data/sanity_problems.jsonl",
        "--output_path",
        "results/mock_run.jsonl",
        "--mock",
        "--metadata_json",
        "results/metadata_run.json",
    ],
)


In [ ]:
# Cell 1d: Pre-flight dataset validation
run_step(
    "Pre-flight: validate main dataset (30 problems)",
    [
        "python",
        "eval/validate_dataset.py",
        "data/problems.jsonl",
        "--schema",
        "problems",
        "--min_examples",
        "30",
        "--max_examples",
        "30",
    ],
)


## 2. Sanity Baseline Evaluation
Verify the baseline performance on the 5-task sanity dataset.

In [ ]:
run_step(
    "Baseline sanity evaluation",
    [
        "python",
        "eval/evaluate_baseline.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--problems_path",
        "data/sanity_problems.jsonl",
        "--output_path",
        "results/phase2_sanity_baseline.jsonl",
        "--metadata_json",
        "results/metadata_run.json",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)


## 2b. Main Baseline Evaluation
Evaluate baseline performance on the full 30-problem dataset.

In [ ]:
run_step(
    "Baseline main evaluation",
    [
        "python",
        "eval/evaluate_baseline.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--problems_path",
        "data/problems.jsonl",
        "--output_path",
        "results/phase2_main_baseline.jsonl",
        "--metadata_json",
        "results/metadata_run.json",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)


## 3. Fine-Tuning (Recipe A)
Execute training using the standard QLoRA recipe.


In [ ]:
run_step(
    "Recipe A QLoRA training",
    [
        "python",
        "training/sft_lora.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--dataset_path",
        "data/thinkanywhere_sft_v2.jsonl",
        "--output_dir",
        "outputs/phase2_recipe_a",
        "--load_in_4bit",
        "--bnb_4bit_compute_dtype",
        "float16",
        "--bnb_4bit_quant_type",
        "nf4",
        "--max_steps",
        "100",
        "--limit_samples",
        "100",
        "--max_seq_length",
        "1024",
        "--lora_rank",
        "8",
        "--lora_alpha",
        "32",
        "--lora_dropout",
        "0.05",
        "--learning_rate",
        "2e-4",
        "--per_device_train_batch_size",
        "1",
        "--gradient_accumulation_steps",
        "4",
    ],
)


## 4. Adapter Probing
Test adapter attachment and output divergence after Recipe A training.


In [ ]:
run_step(
    "Adapter probe",
    [
        "python",
        "eval/adapter_probe.py",
        "--adapter_path",
        "outputs/phase2_recipe_a",
    ],
)


## 5. Evaluation & Reporting
Compare fine-tuned vs baseline for both sanity and main datasets.


In [ ]:
import os

ADAPTER_PATH = "outputs/phase2_recipe_a"

if not os.path.exists(f"{ADAPTER_PATH}/adapter_config.json"):
    raise RuntimeError(
        f"HARD STOP: adapter_config.json not found at {ADAPTER_PATH}. "
        "Run Recipe A training first and verify it completed successfully."
    )

run_step(
    "Fine-tuned sanity evaluation",
    [
        "python",
        "eval/evaluate_finetuned.py",
        "--base_model",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--adapter_path",
        ADAPTER_PATH,
        "--problems_path",
        "data/sanity_problems.jsonl",
        "--output_path",
        "results/phase2_sanity_finetuned.jsonl",
        "--metadata_json",
        "results/metadata_run.json",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)

run_step(
    "Fine-tuned main evaluation",
    [
        "python",
        "eval/evaluate_finetuned.py",
        "--base_model",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--adapter_path",
        ADAPTER_PATH,
        "--problems_path",
        "data/problems.jsonl",
        "--output_path",
        "results/phase2_main_finetuned.jsonl",
        "--metadata_json",
        "results/metadata_run.json",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)

run_step(
    "Generate Sanity Report",
    [
        "python",
        "eval/make_report.py",
        "--baseline",
        "results/phase2_sanity_baseline.jsonl",
        "--finetuned",
        "results/phase2_sanity_finetuned.jsonl",
        "--out",
        "results/phase2_sanity_report.md",
    ],
)

run_step(
    "Generate Main Report",
    [
        "python",
        "eval/make_report.py",
        "--baseline",
        "results/phase2_main_baseline.jsonl",
        "--finetuned",
        "results/phase2_main_finetuned.jsonl",
        "--out",
        "results/phase2_main_report.md",
    ],
)

from IPython.display import Markdown
display(Markdown(open("results/phase2_sanity_report.md").read()))
display(Markdown(open("results/phase2_main_report.md").read()))


## 6. Drive Export
Persist results to Google Drive.

In [ ]:
from pathlib import Path
import shutil, time
from google.colab import drive

MOUNT_POINT = '/content/drive'

def mount_drive(max_retries=2):
    last_err = None
    for i in range(max_retries):
        try:
            drive.mount(MOUNT_POINT, force_remount=True)
            return
        except Exception as e:
            last_err = e
            print(f'[mount attempt {i+1}/{max_retries}] failed: {e}')
            time.sleep(2)

    try:
        drive.mount(MOUNT_POINT)
        return
    except Exception as e:
        raise RuntimeError('HARD STOP: Google Drive mount failed after retries. Restart runtime and mount manually from Files sidebar.') from e

mount_drive()

EXPORT_DIR = Path('/content/drive/MyDrive/codepause_phase2')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for source in [Path('outputs/phase2_recipe_a'), Path('results')]:
    if not source.exists():
        raise RuntimeError(f'HARD STOP: expected artifact path missing: {source}')
    target = EXPORT_DIR / source.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(source, target)
    print(f'Exported {source} -> {target}')

print(f'Phase 2 artifacts persisted to {EXPORT_DIR}')


## 7. Phase 2 Artifact Summary for Orchestrator
Run this cell after Drive/export sync and paste the output back to the orchestrator.


In [ ]:
# Phase 2 artifact summary for orchestrator
from pathlib import Path
import json

candidates = [
    Path('/content/drive/MyDrive/codepause_phase2_v2'),
    Path('/content/drive/MyDrive/codepause_phase2'),
    Path('drive/codepause_phase2_v2'),
    Path('drive/codepause_phase2_v1'),
]

root = next((p for p in candidates if p.exists()), None)
if root is None:
    raise RuntimeError(f'HARD STOP: no Phase 2 Drive export found. Checked: {[str(p) for p in candidates]}')

results = root / 'results'
adapter = root / 'phase2_recipe_a'
required = {
    'main_report': results / 'phase2_main_report.md',
    'main_baseline': results / 'phase2_main_baseline.jsonl',
    'main_finetuned': results / 'phase2_main_finetuned.jsonl',
    'metadata': results / 'metadata_run.json',
    'adapter_config': adapter / 'adapter_config.json',
    'adapter_model': adapter / 'adapter_model.safetensors',
}

def count_jsonl(path):
    if not path.exists():
        return None
    return sum(1 for line in path.read_text(encoding='utf-8').splitlines() if line.strip())

print('PHASE 2 ARTIFACT SUMMARY FOR ORCHESTRATOR')
print(f'root: {root}')
print(f'results_dir: {results}')
print(f'adapter_dir: {adapter}')
print('\nrequired artifacts:')
for name, path in required.items():
    print(f"- {name}: {path} :: {'OK' if path.exists() else 'MISSING'}")

print('\nrecord counts:')
print(f"- baseline records: {count_jsonl(required['main_baseline'])}")
print(f"- finetuned records: {count_jsonl(required['main_finetuned'])}")

if required['metadata'].exists():
    metadata = json.loads(required['metadata'].read_text(encoding='utf-8'))
    print('\nmetadata highlights:')
    for key in ['phase', 'model_name', 'hardware', 'git_sha', 'notebook']:
        print(f'- {key}: {metadata.get(key)}')

if required['main_report'].exists():
    report = required['main_report'].read_text(encoding='utf-8')
    print('\nreport excerpt:')
    print(report[:1200])

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise RuntimeError(f'HARD STOP: missing Phase 2 artifacts: {missing}')

print('\nSTATUS: Phase 2 Drive export is ready for Phase 3 inspection.')
